In [210]:
import pandas as pd 

In [211]:
df_cc_calls = pd.read_csv('../../dataset/original/cc_calls.csv')
print(df_cc_calls.info)

<bound method DataFrame.info of          Contact_ID   Call_Date  Direction cc_care_package  \
0      6.255130e+11  08-05-2025  OUT_BOUND        Standard   
1      5.910870e+11  25-11-2024  OUT_BOUND        Standard   
2      5.650910e+11  23-10-2024   IN_BOUND        Standard   
3      5.939750e+11  13-01-2025   IN_BOUND         Premier   
4      6.222820e+11  19-03-2025   IN_BOUND        Standard   
...             ...         ...        ...             ...   
32877  5.963620e+11  17-02-2025  OUT_BOUND   Not Discussed   
32878  6.236330e+11  08-04-2025   IN_BOUND   Not Discussed   
32879  6.772340e+11  01-07-2025   IN_BOUND   Not Discussed   
32880  5.928980e+11  23-12-2024  OUT_BOUND   Not Discussed   
32881  5.933910e+11  03-01-2025  OUT_BOUND   Not Discussed   

      cc_care_package_discussed cc_urgency_getting_on_site  \
0                           Yes                         No   
1                           Yes                         No   
2                           Yes      

## Handling duplicate cols 

In [212]:
print( "Before duplication removal:",df_cc_calls.duplicated().sum())
df_cc_calls = df_cc_calls.drop_duplicates()
print( "After duplication removal:",df_cc_calls.duplicated().sum())

print("Total rows after duplication removal:", len(df_cc_calls))


Before duplication removal: 93
After duplication removal: 0
Total rows after duplication removal: 32789


## Fixing Datatypes and standardising

In [213]:
df_validation = pd.read_csv("../../dataset/01_raw/cc_calls/data_validation_summary.csv")
print(df_validation.info)

<bound method DataFrame.info of     Unnamed: 0                               column_name data_type  \
0            0                                Contact_ID   float64   
1            1                                 Call_Date    object   
2            2                                 Direction    object   
3            3                           cc_care_package    object   
4            4                 cc_care_package_discussed    object   
5            5                cc_urgency_getting_on_site    object   
6            6                    cc_external_consultant    object   
7            7               cc_agent_cross_sell_attempt    object   
8            8               cc_customer_issues_concerns    object   
9            9  cc_business_struggles_financial_hardship    object   
10          10                      cc_call_initiated_by    object   
11          11               cc_questionnaire_completion    object   
12          12                       cc_chasing_response  

In [214]:
df_mixed = pd.read_csv("../../dataset/01_raw/cc_calls/mixed_data_types.csv")
print(df_mixed)

                                  column  numeric_count  non_numeric_count  \
0    cc_contractor_sentiment_start_score          32638                149   
1      cc_contractor_sentiment_end_score          32639                148   
2  cc_contractor_sentiment_overall_score          27896               4891   
3   cc_contractor_sentiment_issues_score          11376              21411   
4                   cc_pricing_mentioned             40              32747   
5            cc_pricing_sentiment_impact             17              32770   
6                    cc_refund_discussed              8              32779   

   sample_numeric                                 sample_non_numeric  
0              20                                            Neutral  
1              30   which was caused by an expired SSIP provider ...  
2              30                                      Not Discussed  
3              20                                      Not Discussed  
4              80   

In [215]:
df_cc_calls['cc_contractor_sentiment_start_score'].unique()

array(['20', '0', '50', '40', nan, 'Neutral', 'Not Discussed',
       'Satisfied', ' investigated and resolved the issue',
       'Dissatisfied', '80', ' Chris', ' Josh', '60', '10',
       ' provided updates and reassurance', '30',
       ' transferred the call to Athena', ' Owen',
       ' assisted Ben and contacted Kelly', ' and the agent',
       ' and the agent agreed to update the application and notify the primary contact once it\'s approved."',
       '70',
       ' assisted Amy in resolving the issue by chasing up with the audit team and adding Amy\'s mobile number to the contacts."',
       ' assisted with the request and provided her email address to receive the request via email."',
       ' attempted to reset the password but was unable to do so and reported the issue to the IT department."',
       ' assisted Kisra in resolving the issue and provided guidance on completing the outstanding parts of the questionnaire."',
       ' guided Chris through the process of completi

In [216]:
df_cc_calls['cc_contractor_sentiment_start_score'] = (
    df_cc_calls['cc_contractor_sentiment_start_score']
    .astype(str)
    .str.strip()
    .str.lower()
)

In [217]:
df_cc_calls['cc_contractor_sentiment_start_score'] = (
    df_cc_calls['cc_contractor_sentiment_start_score']
    .replace('nan', 'unknown')
)

In [218]:
def clean_sentiment(val):
    # numeric scores
    if val.isdigit():
        return str(val)
    
    # sentiment labels
    elif 'satisfied' in val:
        return 'satisfied'
    elif 'dissatisfied' in val:
        return 'dissatisfied'
    elif 'neutral' in val:
        return 'neutral'
    elif 'not discussed' in val:
        return 'not discussed'
    
    # everything else = noise
    else:
        return 'unknown'


df_cc_calls['cc_contractor_sentiment_start_score'] = df_cc_calls['cc_contractor_sentiment_start_score'].apply(clean_sentiment)

In [219]:
df_cc_calls['cc_contractor_sentiment_start_score'].unique()

array(['20', '0', '50', '40', 'unknown', 'neutral', 'not discussed',
       'satisfied', '80', '60', '10', '30', '70', '90', '100'],
      dtype=object)

# ==== - -= =- 

In [220]:
df_cc_calls['cc_contractor_sentiment_end_score'].unique()

array(['30', '0', '60', '40', '80', '20', nan, '50', '10', '90',
       ' which was caused by an expired SSIP provider claim."',
       ' helped him understand the situation and advised him to follow up with the customer care advisor',
       ' helped her investigate and suggested it was likely paid by credit card',
       '70', ' resolving the customer\'s concerns."', '100',
       'Not Discussed', '95', ' who assisted with the issue."',
       ' assisted with the process and informed the customer of a potential £60 charge."',
       ' who was able to remove the category."', ' Gabrielle',
       'Satisfied', 'Neutral', 'Dissatisfied', '75'], dtype=object)

In [221]:
df_cc_calls['cc_contractor_sentiment_end_score'] = (
    df_cc_calls['cc_contractor_sentiment_end_score']
    .astype(str)
    .str.strip()
    .str.lower()
)

In [222]:
df_cc_calls['cc_contractor_sentiment_end_score'] = (
    df_cc_calls['cc_contractor_sentiment_end_score']
    .replace('nan', 'unknown')
)

In [223]:
def clean_sentiment(val):
    # numeric scores
    if val.isdigit():
        return str(val)
    
    # sentiment labels
    elif 'satisfied' in val:
        return 'satisfied'
    elif 'dissatisfied' in val:
        return 'dissatisfied'
    elif 'neutral' in val:
        return 'neutral'
    elif 'not discussed' in val:
        return 'not discussed'
    
    # everything else = noise
    else:
        return 'unknown'


df_cc_calls['cc_contractor_sentiment_end_score'] = df_cc_calls['cc_contractor_sentiment_end_score'].apply(clean_sentiment)

In [224]:
df_cc_calls['cc_contractor_sentiment_end_score'].unique()

array(['30', '0', '60', '40', '80', '20', 'unknown', '50', '10', '90',
       '70', '100', 'not discussed', '95', 'satisfied', 'neutral', '75'],
      dtype=object)

In [225]:
df_cc_calls['cc_contractor_sentiment_overall_score'].unique()

array(['30', '0', '40', '20', '60', '80', '55', '50', nan, '10', '90',
       'Not Discussed', 'Satisfied', '100', '85', ' Owen Davis."',
       ' but couldn\'t provide further details due to security restrictions."',
       '70', '45', '65', '35', '75', '95', '25',
       ' guided him through the process and offered to add him as a secondary contact to the account."',
       '92'], dtype=object)

In [226]:
df_cc_calls['cc_contractor_sentiment_overall_score'] = (
    df_cc_calls['cc_contractor_sentiment_overall_score']
    .astype(str)
    .str.strip()
    .str.lower()
)

In [227]:
df_cc_calls['cc_contractor_sentiment_overall_score'] = (
    df_cc_calls['cc_contractor_sentiment_overall_score']
    .replace('nan', 'unknown')
)

In [228]:
def clean_overall(val):
    # numeric scores
    if val.isdigit():
        return str(val)
    
    # valid categories
    elif 'satisfied' in val:
        return 'satisfied'
    elif 'not discussed' in val:
        return 'not discussed'
    
    # everything else = noise
    else:
        return 'unknown'


df_cc_calls['cc_contractor_sentiment_overall_score'] = \
    df_cc_calls['cc_contractor_sentiment_overall_score'].apply(clean_overall)

In [229]:
df_cc_calls['cc_contractor_sentiment_overall_score'].unique()

array(['30', '0', '40', '20', '60', '80', '55', '50', 'unknown', '10',
       '90', 'not discussed', 'satisfied', '100', '85', '70', '45', '65',
       '35', '75', '95', '25', '92'], dtype=object)

In [230]:
df_cc_calls['cc_contractor_sentiment_issues_score'].unique()

array(['20', '0', '30', '10', '60', '40', nan, '50', '90', '80',
       'Not Discussed', '100', '70', 'Neutral', '75', '95', '65', '55',
       '85', 'Satisfied', '92', 'Not Discussed]'], dtype=object)

In [231]:
df_cc_calls['cc_contractor_sentiment_issues_score'] = (
    df_cc_calls['cc_contractor_sentiment_issues_score']
    .astype(str)
    .str.strip()
    .str.lower()
)

In [232]:
df_cc_calls['cc_contractor_sentiment_issues_score'] = (
    df_cc_calls['cc_contractor_sentiment_issues_score']
    .replace('nan', 'unknown')
)

In [233]:
def clean_overall(val):
    
    if val.isdigit():
        return str(val)
    
    
    elif 'satisfied' in val:
        return 'satisfied'
    elif 'not discussed' in val:
        return 'not discussed'
    
    
    else:
        return 'unknown'


df_cc_calls['cc_contractor_sentiment_issues_score'] = \
    df_cc_calls['cc_contractor_sentiment_issues_score'].apply(clean_overall)

In [234]:
df_cc_calls['cc_contractor_sentiment_issues_score'].unique()

array(['20', '0', '30', '10', '60', '40', 'unknown', '50', '90', '80',
       'not discussed', '100', '70', '75', '95', '65', '55', '85',
       'satisfied', '92'], dtype=object)

In [235]:
df_cc_calls['cc_pricing_mentioned'].unique()

array(['Yes', 'No', nan, '80', 'Not Discussed', '90', '70', '20', '50',
       '100', '85', '95'], dtype=object)

In [236]:
df_cc_calls['cc_pricing_mentioned'] = (
    df_cc_calls['cc_pricing_mentioned']
    .astype(str)
    .str.strip()
    .str.lower()
)

In [237]:
df_cc_calls['cc_pricing_mentioned'] = (
    df_cc_calls['cc_pricing_mentioned']
    .replace('nan', 'unknown')
)

In [238]:
def clean_pricing(val):
    if val == 'yes':
        return 'yes'
    elif val == 'no':
        return 'no'
    elif 'not discussed' in val:
        return 'no'
    
    elif val.isdigit():
        return 'unknown'
    
    else:
        return 'unknown'

df_cc_calls['cc_pricing_mentioned'] = df_cc_calls['cc_pricing_mentioned'].apply(clean_pricing)

In [239]:
df_cc_calls['cc_pricing_mentioned'].unique()

array(['yes', 'no', 'unknown'], dtype=object)

In [240]:
df_cc_calls['cc_pricing_sentiment_impact'].unique()

array(['Yes', 'No', nan, '80', '70', '95', '90', '85', '20'], dtype=object)

In [241]:
df_cc_calls['cc_pricing_sentiment_impact'] = (
    df_cc_calls['cc_pricing_sentiment_impact']
    .astype(str)
    .str.strip()
    .str.lower()
)


In [242]:
df_cc_calls['cc_pricing_sentiment_impact'] = (
    df_cc_calls['cc_pricing_sentiment_impact']
    .replace('nan', 'unknown')
)

In [243]:
def clean_pricing_impact(val):
    if val == 'yes':
        return 'yes'
    elif val == 'no':
        return 'no'
    
    # numeric → noise
    elif val.isdigit():
        return 'unknown'
    
    else:
        return 'unknown'

df_cc_calls['cc_pricing_sentiment_impact'] =  df_cc_calls['cc_pricing_sentiment_impact'].apply(clean_pricing_impact)

In [244]:
df_cc_calls['cc_pricing_sentiment_impact'].unique()

array(['yes', 'no', 'unknown'], dtype=object)

In [245]:
df_cc_calls['cc_refund_discussed'   ].unique()

array(['No', 'Yes', nan, '80', '60', '90'], dtype=object)

In [246]:
df_cc_calls['cc_refund_discussed'] = (
    df_cc_calls['cc_refund_discussed']
    .astype(str)
    .str.strip()
    .str.lower()
)

In [247]:
df_cc_calls['cc_refund_discussed'] = (
    df_cc_calls['cc_refund_discussed']
    .replace('nan', 'unknown')
)

In [248]:
def clean_refund(val):
    if val == 'yes':
        return 'yes'
    elif val == 'no':
        return 'no'
    
    # numeric noise
    elif val.isdigit():
        return 'unknown'
    
    else:
        return 'unknown'

df_cc_calls['cc_refund_discussed'] = \
    df_cc_calls['cc_refund_discussed'].apply(clean_refund)

In [249]:
df_cc_calls['cc_refund_discussed'   ].unique()

array(['no', 'yes', 'unknown'], dtype=object)

## Analyzing Cols with more NULLs 

In [250]:
print(len(df_cc_calls))
# print(df_cc_calls.isnull().sum())

print(len(df_cc_calls) - df_cc_calls.isnull().sum())

32789
Contact_ID                                  32789
Call_Date                                   32789
Direction                                   32789
cc_care_package                             32652
cc_care_package_discussed                   32652
cc_urgency_getting_on_site                  32652
cc_external_consultant                      32652
cc_agent_cross_sell_attempt                 32652
cc_customer_issues_concerns                 32652
cc_business_struggles_financial_hardship    32652
cc_call_initiated_by                        32652
cc_questionnaire_completion                 32757
cc_chasing_response                         32757
cc_issues_within_questionnaire              32323
cc_login_issues                             32756
cc_platform_issues                          32756
cc_dissatisfaction_time_to_complete         32756
cc_process_complexity_concerns              32751
cc_questions_harder_than_expected           32752
cc_dissatisfaction_support                  

In [251]:
print(
    df_validation[['column_name', 'column_type', 'null_percentage']]
    .sort_values(by='null_percentage', ascending=False)
)

                                 column_name  column_type  null_percentage
30                                    Co_Ref  categorical             3.64
13            cc_issues_within_questionnaire  categorical             1.42
3                            cc_care_package  categorical             0.42
4                  cc_care_package_discussed  categorical             0.42
5                 cc_urgency_getting_on_site  categorical             0.42
6                     cc_external_consultant  categorical             0.42
7                cc_agent_cross_sell_attempt  categorical             0.42
8                cc_customer_issues_concerns  categorical             0.42
9   cc_business_struggles_financial_hardship  categorical             0.42
10                      cc_call_initiated_by  categorical             0.42
28               cc_contractor_suggest_leave  categorical             0.29
24      cc_contractor_sentiment_issues_score  categorical             0.29
21       cc_contractor_se

In [252]:

cols_to_drop = df_validation[
    df_validation['null_percentage'] > 90
]['column_name'].tolist()





print("Before dropping columns:", len(df_cc_calls.columns))
print("Columns to drop:", cols_to_drop)


df_cc_calls = df_cc_calls.drop(columns=cols_to_drop, errors='ignore')

print("After dropping columns:", len(df_cc_calls.columns))

Before dropping columns: 33
Columns to drop: []
After dropping columns: 33


## ================================================================

In [253]:
df_cc_calls.to_csv('../../dataset/02_basic_clean/cc_calls.csv', index=False)